In [ ]:
import os
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, utils, models
from PIL import Image
import copy
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, precision_recall_curve, roc_auc_score
from sklearn.metrics import classification_report
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold, KFold
from collections import defaultdict, Counter
from facenet_pytorch import InceptionResnetV1
from models_vggface.vgg_face import vgg_face, VggFaceFeatureExtractor
import clip


In [ ]:
class RFDDataset(Dataset):
    def __init__(self, img_paths, disease_labels, transform=None):
        self.img_paths = img_paths
        self.disease_labels = disease_labels
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        label = self.disease_labels[idx]
        image = Image.open(img_path).convert("RGB")  # Convert to RGB

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
def prepare_df(topk):
    # Load metadata
    img_dir = "data/rd_images"
    data = pd.read_csv("data/disease_images.csv")

    img_names = data["image_name"]
    # disease_names = data["disease_name"]
    disease_abbr = data["disease_abbr"]

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    img_paths = [os.path.join(img_dir, disease, name) for disease, name in zip(disease_abbr, img_names)]
    disease_mappings = {disease: i for i, disease in enumerate(disease_abbr.unique())}
    disease_labels = [disease_mappings[disease] for disease in disease_abbr]

    '''
    Below is to make sure that classes with single sample will be placed into training set for sure
    '''

    class_to_indices = defaultdict(list)
    for idx, label in enumerate(disease_labels):
        class_to_indices[label].append(idx)

    singleton_indices = [idxs[0] for idxs in class_to_indices.values() if len(idxs) == 1]
    non_singleton_indices = [idx for idxs in class_to_indices.values() if len(idxs) > 1 for idx in idxs]

    # Create the singleton part of training set
    singleton_imgs = [img_paths[i] for i in singleton_indices]
    singleton_labels = [disease_labels[i] for i in singleton_indices]

    # For non-singleton samples, split into train/test (stratified)
    non_singleton_imgs = [img_paths[i] for i in non_singleton_indices]
    non_singleton_labels = [disease_labels[i] for i in non_singleton_indices]

    train_ns_imgs, test_imgs, train_ns_labels, test_labels = train_test_split(
        non_singleton_imgs,
        non_singleton_labels,
        test_size=0.25,
        random_state=42,
        stratify=non_singleton_labels
    )

    # training set = singleton + non-singleton training
    train_imgs = singleton_imgs + train_ns_imgs
    train_labels = singleton_labels + train_ns_labels

    # add data augmentation (synthetic data) towards training set
    synthetic_dir = "FastGAN-pytorch/output_log_train/train_results/test1"
    synthetic_meta_csv = f"data_aug/synthetic/meta_new/fastgan_meta/top_{topk}_cos_metadata.csv"
    synthetic_img_paths = []
    synthetic_labels = []
    topk_csv_file = pd.read_csv(synthetic_meta_csv)
    topk_image_names = topk_csv_file["image_name"]
    disease_labels = topk_csv_file['label']
    synthetic_img_paths = [os.path.join(synthetic_dir, image_name) for image_name in topk_image_names]
    synthetic_labels = [disease_mappings[class_name] for class_name in disease_labels]

    train_imgs = synthetic_img_paths + train_imgs
    train_labels = synthetic_labels + train_labels

    # can also add dreambooth samples for data augmentation
    # add data augmentation (synthetic data) towards training set
    # synthetic_dir = "data_aug/synthetic/synthetic_train_realistic"
    # synthetic_meta_csv = f"data_aug/synthetic/meta_new/synthetic_train_realistic_topk_meta/top_{topk}_cos_metadata.csv"
    # synthetic_img_paths = []
    # synthetic_labels = []
    # topk_csv_file = pd.read_csv(synthetic_meta_csv)
    # topk_image_names = topk_csv_file["image_name"]
    # topk_class_names = topk_image_names.apply(lambda x: x.split("_")[0]).tolist()
    # synthetic_img_paths = [os.path.join(synthetic_dir, class_name, image_name) for class_name, image_name in zip(topk_class_names, topk_image_names)]
    # synthetic_labels = [disease_mappings[class_name] for class_name in topk_class_names]

    # train_imgs = synthetic_img_paths + train_imgs
    # train_labels = synthetic_labels + train_labels

    # Prepare data for k-fold on training set
    train_indices = list(range(len(train_imgs)))
    label_array = np.array(train_labels)

    # Group by class
    class_to_indices = defaultdict(list)
    for idx, label in enumerate(label_array):
        class_to_indices[label].append(idx)

    # Singleton detection
    singleton_indices = [idxs[0] for idxs in class_to_indices.values() if len(idxs) == 1]
    non_singleton_indices = [idx for idxs in class_to_indices.values() if len(idxs) > 1 for idx in idxs]
    non_singleton_labels = [train_labels[idx] for idx in non_singleton_indices]

    # StratifiedKFold on non-singleton training data
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    folds = []
    for train_idx_ns, val_idx_ns in skf.split(non_singleton_indices, non_singleton_labels):
        train_idx = [non_singleton_indices[i] for i in train_idx_ns] + singleton_indices
        val_idx = [non_singleton_indices[i] for i in val_idx_ns]
        folds.append((train_idx, val_idx))

    # data sets
    train_dataset = RFDDataset(train_imgs, train_labels, transform=transform)
    test_dataset = RFDDataset(test_imgs, test_labels, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4, drop_last=False)

    return disease_mappings, folds, train_dataset, test_dataset, test_loader

In [ ]:
_, _, train_dataset, test_dataset, test_loader = prepare_df(1000)
print(test_loader.dataset.__len__())
print(len(train_dataset))

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class RareDiseaseModel(nn.Module):
    def __init__(self, backbone, num_classes, use_se=True, dropout_rate=0.4, use_layernorm=False, conv_out_channels=None):
        """
        Args:
            backbone (nn.Module): Any feature extractor (EfficientNet, ResNet, ViT, etc.).
            num_classes (int): Number of output classes.
            use_se (bool): Whether to use Squeeze-and-Excitation (SE) block.
            dropout_rate (float): Dropout probability.
            use_layernorm (bool): Use LayerNorm instead of BatchNorm to avoid batch size issues.
        """
        super(RareDiseaseModel, self).__init__()

        self.backbone = backbone  # Store backbone
        self.use_se = use_se
        self.se_block_2d = None
        self.se_block_4d = None
        self.in_features = self._get_backbone_out_features()

        # Optional SE
        if use_se:
            self.se_block_2d = nn.Sequential(
                nn.Linear(self.in_features, self.in_features // 16),
                nn.ReLU(),
                nn.Linear(self.in_features // 16, self.in_features),
                nn.Sigmoid()
            )

            # Only define 4D SE block if conv_out_channels is provided
            if conv_out_channels is not None:
                self.se_block_4d = nn.Sequential(
                    nn.AdaptiveAvgPool2d(1),
                    nn.Conv2d(conv_out_channels, conv_out_channels // 16, kernel_size=1),
                    nn.ReLU(),
                    nn.Conv2d(conv_out_channels // 16, conv_out_channels, kernel_size=1),
                    nn.Sigmoid()
                )
            else:
                self.se_block_4d = None


        # Optional normalization
        norm_layer = nn.LayerNorm(self.in_features) if use_layernorm else nn.BatchNorm1d(self.in_features)

        # Classification head
        self.classifier = nn.Sequential(
            norm_layer,
            nn.Dropout(dropout_rate),
            nn.Linear(self.in_features, num_classes)
        )

    def _get_backbone_out_features(self):
        param = next(self.backbone.parameters())
        dummy_input = torch.randn(1, 3, 224, 224).to(device=param.device, dtype=param.dtype)

        with torch.no_grad():
            # ResNet-style
            if hasattr(self.backbone, 'fc') and isinstance(self.backbone.fc, nn.Linear):
                in_features = self.backbone.fc.in_features
                self.backbone.fc = nn.Identity()
                return in_features

            # CLIP ViT model (clip_model.visual)
            if type(self.backbone).__name__ == "VisionTransformer" and "clip" in str(type(self.backbone)).lower():
                features = self.backbone(dummy_input)
                return features.shape[1]

            # VGGFace (your custom conv feature extractor)
            if isinstance(self.backbone, VggFaceFeatureExtractor):
                out = self.backbone(dummy_input)
                out = torch.flatten(out, 1)
                return out.shape[1]

            # EfficientNet, DenseNet, Inception, etc.
            if hasattr(self.backbone, 'classifier'):
                classifier = self.backbone.classifier
                if isinstance(classifier, nn.Sequential):
                    for layer in reversed(classifier):
                        if isinstance(layer, nn.Linear):
                            in_features = layer.in_features
                            self.backbone.classifier = nn.Identity()
                            return in_features
                elif isinstance(classifier, nn.Linear):
                    in_features = classifier.in_features
                    self.backbone.classifier = nn.Identity()
                    return in_features
                else:
                    raise TypeError("Unknown classifier structure in `.classifier`")

            # Swin
            if hasattr(self.backbone, 'head') and isinstance(self.backbone.head, nn.Linear):
                in_features = self.backbone.head.in_features
                self.backbone.head = nn.Identity()
                return in_features

            # ViT from timm
            if hasattr(self.backbone, 'heads') and hasattr(self.backbone.heads, 'head'):
                in_features = self.backbone.heads.head.in_features
                self.backbone.heads.head = nn.Identity()
                return in_features

            # Fallback: forward + flatten
            features = self.backbone(dummy_input)
            if isinstance(features, tuple):
                features = features[0]
            if features.ndim == 4:
                features = torch.flatten(features, 1)
            return features.shape[1]

        # If nothing matches
        raise ValueError(f"Unsupported backbone architecture: {type(self.backbone)}")


    def forward(self, x):
        if hasattr(self.backbone, "encode_image"):
            features = self.backbone.encode_image(x)
        else:
            features = self.backbone(x)

        if isinstance(features, tuple):
            features = features[0]
        elif hasattr(features, 'logits'):
            features = features.logits

        # --- Apply SE block if needed ---
        if features.ndim == 4 and self.se_block_4d:
            se_weight = self.se_block_4d(features)
            features = features * se_weight
        elif features.ndim == 2 and self.se_block_2d:
            se_weight = self.se_block_2d(features)
            features = features * se_weight

        # --- Flatten if needed ---
        if features.ndim == 4:
            features = torch.flatten(features, 1)

        return self.classifier(features)



In [ ]:
def train(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_labels = []
    all_preds = []

    with tqdm(train_loader, unit="batch") as bar:
        for images, labels in bar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            bar.set_postfix(loss=loss.item())

    accuracy = accuracy_score(all_labels, all_preds) * 100

    return total_loss / len(train_loader), accuracy

def topk_accuracy(output, target, k=5):
    with torch.no_grad():
        topk_preds = torch.topk(output, k, dim=1).indices  # Get top-K class indices
        correct = topk_preds.eq(target.view(-1, 1).expand_as(topk_preds))  # Check if true label is in top-K
        return correct.any(dim=1).float().mean().item()  # Compute mean accuracy

def evaluate(model, test_loader, criterion, device, topk=[1, 5, 10, 30]):
    model.eval()
    total_loss = 0
    all_labels = []
    all_preds = []
    all_probs = []
    topk_accuracies = {k: 0 for k in topk}

    with torch.no_grad():
        for images, labels in tqdm(test_loader, unit="batch"):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, predicted = outputs.max(1)

            probabilities = F.softmax(outputs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probabilities.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    accuracy = accuracy_score(all_labels, all_preds) * 100
    f1 = f1_score(all_labels, all_preds, average="macro")

    # Top-K Accuracies (global, consistent with sklearn)
    all_outputs = torch.tensor(all_probs)
    all_labels_tensor = torch.tensor(all_labels)
    topk_accuracies = {k: topk_accuracy(all_outputs, all_labels_tensor, k=k) * 100 for k in topk}

    return total_loss / len(test_loader), accuracy, f1, topk_accuracies, all_labels, all_preds, all_probs


In [ ]:
def train_model_on_folds(base_model, dataset, folds, save_dir, device, num_epochs=30):
    os.makedirs(save_dir, exist_ok=True)
    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(folds):
        print(f"\n Fold {fold + 1}")
        best_acc = 0

        train_loader = DataLoader(Subset(dataset, train_idx), batch_size=32, shuffle=True, num_workers=4, drop_last=False)
        val_loader = DataLoader(Subset(dataset, val_idx), batch_size=32, shuffle=False, num_workers=4, drop_last=False)

        # Initialize model + training config
        model = copy.deepcopy(base_model)
        model.to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

        for epoch in range(num_epochs):
            print(f"\n Fold {fold + 1}, Epoch {epoch + 1}/{num_epochs}")
            train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
            val_loss, val_acc, val_f1, topk_accuracies, _, _, _ = evaluate(model, val_loader, criterion, device)

            scheduler.step()

            # check acc
            if val_acc >= best_acc:
                best_acc = val_acc
                
                print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}% | "
                  f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}%, F1: {val_f1:.4f}")

                print(" | ".join([f"Top-{k} Accuracy: {acc:.4f}%" for k, acc in topk_accuracies.items()]))

                torch.save(model.state_dict(), os.path.join(save_dir, f"fold{fold+1}_best.pth"))
                print(f"Best model saved with acc {best_acc:.4f}%")

        fold_metrics.append({
            "val_acc": best_acc,
            "val_topk": topk_accuracies
        })

    # Summary
    avg_acc = np.mean([m["val_acc"] for m in fold_metrics])
    avg_topk = {k: np.mean([m["val_topk"][k] for m in fold_metrics]) for k in fold_metrics[0]["val_topk"].keys()}

    print(f"\n Average across {len(folds)} folds:")
    print(f" - Accuracy: {avg_acc:.2f}%")
    print(" | ".join([f"Top-{k} Accuracy: {acc:.4f}%" for k, acc in avg_topk.items()]))
    return fold_metrics


In [ ]:
from facenet_pytorch import InceptionResnetV1
import torch
import torch.nn as nn

class RareDiseaseFaceNet(nn.Module):
    def __init__(self, num_classes, freeze_backbone=False, use_se=True, dropout_rate=0.5):
        super(RareDiseaseFaceNet, self).__init__()

        # Load FaceNet (pretrained on VGGFace2)
        self.backbone = InceptionResnetV1(pretrained='vggface2', classify=False)

        # Remove FaceNet’s last batch normalization layer
        self.backbone.last_bn = nn.Identity()  # This prevents the batch norm issue

        # Extract feature size (FaceNet outputs a 512D embedding)
        in_features = self.backbone.last_linear.in_features
        self.backbone.last_linear = nn.Identity()  # Remove default classification layer

        # Optional: Freeze the FaceNet backbone
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # Optional Squeeze-and-Excitation (SE) block for feature refinement
        if use_se:
            self.se_block = nn.Sequential(
                nn.Linear(in_features, in_features // 16),
                nn.ReLU(),
                nn.Linear(in_features // 16, in_features),
                nn.Sigmoid()
            )
        else:
            self.se_block = None

        # Custom classifier for rare disease classification
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(dropout_rate),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)  # Extract FaceNet features
        if self.se_block:
            se_weight = self.se_block(features)
            features = features * se_weight  # Apply SE weighting

        output = self.classifier(features)  # Custom classification head
        return output


In [ ]:
# facenet
cutoffs = [1000, 2000, 4000, 6000]
for x in cutoffs:
    disease_mappings, folds, train_dataset, test_dataset, test_loader = prepare_df(x)
    num_classes = len(disease_mappings)
    save_dir = f"logs_kfold_fastgan_topk_cos/FACENET/top{x}"
    device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
    model = RareDiseaseFaceNet(num_classes=num_classes, freeze_backbone=False, use_se=True, dropout_rate=0.5)
    facenet_matrix = train_model_on_folds(model, train_dataset, folds, save_dir, device, num_epochs=50)

In [ ]:
# facenet test
cutoffs = [1000, 2000, 4000, 6000]
for x in cutoffs:
    disease_mappings, _, _, _, test_loader = prepare_df(x)
    test_metrics = []
    print(f"==========================cutoff {x}==========================")
    for fold in range(5):
        num_classes = len(disease_mappings)
        device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
        test_model = RareDiseaseFaceNet(num_classes=num_classes).to(device)
        test_model.load_state_dict(torch.load(f'logs_kfold_fastgan_topk_cos/FACENET/top{x}/fold{fold+1}_best.pth'))
        test_model.eval()  # Set model to evaluation mode
        criterion = nn.CrossEntropyLoss()
        val_loss, val_acc, val_f1, topk_accuracies, _, _, _ = evaluate(test_model, test_loader, criterion, device)
        print(val_acc)
        print(" | ".join([f"Top-{k} Accuracy: {acc:.4f}%" for k, acc in topk_accuracies.items()]))

        test_metrics.append({
            "val_acc": val_acc,
            "val_topk": topk_accuracies
        })

    # Compute average and std of overall accuracy
    val_accs = [m["val_acc"] for m in test_metrics]
    avg_acc = np.mean(val_accs)
    std_acc = np.std(val_accs, ddof=1)

    # Compute average and std for each top-k accuracy
    avg_topk = {}
    std_topk = {}
    for k in test_metrics[0]["val_topk"].keys():
        scores = [m["val_topk"][k] for m in test_metrics]
        avg_topk[k] = np.mean(scores)
        std_topk[k] = np.std(scores, ddof=1)

    # Print results
    print(f"\nAverage across {len(test_metrics)} folds:")
    print(f" - Accuracy: {avg_acc:.2f}% ± {std_acc:.2f}% (1-sigma)")

    for k in sorted(avg_topk.keys(), key=int):
        print(f" - Top-{k} Accuracy: {avg_topk[k]:.2f}% ± {std_topk[k]:.2f}% (1-sigma)")


In [ ]:
# resnet152
cutoffs = [1000, 2000, 4000, 6000]
for x in cutoffs:
    disease_mappings, folds, train_dataset, test_dataset, test_loader = prepare_df(x)
    num_classes = len(disease_mappings)
    save_dir = f"logs_kfold_fastgan_topk_cos/RESNET152/top{x}"
    device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
    backbone = models.resnet152(weights='ResNet152_Weights.IMAGENET1K_V2')
    model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False)
    train_model_on_folds(model, train_dataset, folds, save_dir, device, num_epochs=50)



In [ ]:
# resnet152 test
cutoffs = [1000, 2000, 4000, 6000]
for x in cutoffs:
    disease_mappings, _, _, _, test_loader = prepare_df(x)
    test_metrics = []
    print(f"==========================cutoff {x}==========================")
    for fold in range(5):
        num_classes = len(disease_mappings)
        device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
        backbone = models.resnet152(weights='ResNet152_Weights.IMAGENET1K_V2')
        test_model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False)
        test_model.load_state_dict(torch.load(f'logs_kfold_fastgan_topk_cos/RESNET152/top{x}/fold{fold+1}_best.pth'))
        test_model.to(device)
        test_model.eval()  # Set model to evaluation mode
        criterion = nn.CrossEntropyLoss()
        val_loss, val_acc, val_f1, topk_accuracies, _, _, _ = evaluate(test_model, test_loader, criterion, device)
        print(val_acc)
        print(" | ".join([f"Top-{k} Accuracy: {acc:.4f}%" for k, acc in topk_accuracies.items()]))

        test_metrics.append({
            "val_acc": val_acc,
            "val_topk": topk_accuracies
        })

    # Compute average and std of overall accuracy
    val_accs = [m["val_acc"] for m in test_metrics]
    avg_acc = np.mean(val_accs)
    std_acc = np.std(val_accs, ddof=1)

    # Compute average and std for each top-k accuracy
    avg_topk = {}
    std_topk = {}
    for k in test_metrics[0]["val_topk"].keys():
        scores = [m["val_topk"][k] for m in test_metrics]
        avg_topk[k] = np.mean(scores)
        std_topk[k] = np.std(scores, ddof=1)

    # Print results
    print(f"\nAverage across {len(test_metrics)} folds:")
    print(f" - Accuracy: {avg_acc:.2f}% ± {std_acc:.2f}% (1-sigma)")

    for k in sorted(avg_topk.keys(), key=int):
        print(f" - Top-{k} Accuracy: {avg_topk[k]:.2f}% ± {std_topk[k]:.2f}% (1-sigma)")

In [ ]:
# densenet169
cutoffs = [1000, 2000, 4000, 6000]
for x in cutoffs:
    disease_mappings, folds, train_dataset, test_dataset, test_loader = prepare_df(x)
    num_classes = len(disease_mappings)
    save_dir = f"logs_kfold_fastgan_topk_cos/DENSENET169/top{x}"
    device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
    backbone = models.densenet169(weights='DenseNet169_Weights.DEFAULT')
    model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False)
    train_model_on_folds(model, train_dataset, folds, save_dir, device, num_epochs=50)

In [ ]:
# densenet169 test
cutoffs = [1000, 2000, 4000, 6000]
for x in cutoffs:
    disease_mappings, _, _, _, test_loader = prepare_df(x)
    test_metrics = []
    print(f"==========================cutoff {x}==========================")
    for fold in range(5):
        num_classes = len(disease_mappings)
        device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
        backbone = models.densenet169(weights='DenseNet169_Weights.DEFAULT')
        test_model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False)
        test_model.load_state_dict(torch.load(f'logs_kfold_fastgan_topk_cos/DENSENET169/top{x}/fold{fold+1}_best.pth'))
        test_model.to(device)
        test_model.eval()  # Set model to evaluation mode
        criterion = nn.CrossEntropyLoss()
        val_loss, val_acc, val_f1, topk_accuracies, _, _, _ = evaluate(test_model, test_loader, criterion, device)
        print(val_acc)
        print(" | ".join([f"Top-{k} Accuracy: {acc:.4f}%" for k, acc in topk_accuracies.items()]))

        test_metrics.append({
            "val_acc": val_acc,
            "val_topk": topk_accuracies
        })

    # Compute average and std of overall accuracy
    val_accs = [m["val_acc"] for m in test_metrics]
    avg_acc = np.mean(val_accs)
    std_acc = np.std(val_accs, ddof=1)

    # Compute average and std for each top-k accuracy
    avg_topk = {}
    std_topk = {}
    for k in test_metrics[0]["val_topk"].keys():
        scores = [m["val_topk"][k] for m in test_metrics]
        avg_topk[k] = np.mean(scores)
        std_topk[k] = np.std(scores, ddof=1)

    # Print results
    print(f"\nAverage across {len(test_metrics)} folds:")
    print(f" - Accuracy: {avg_acc:.2f}% ± {std_acc:.2f}% (1-sigma)")

    for k in sorted(avg_topk.keys(), key=int):
        print(f" - Top-{k} Accuracy: {avg_topk[k]:.2f}% ± {std_topk[k]:.2f}% (1-sigma)")

In [ ]:
# SWIN_T
cutoffs = [1000, 2000, 4000, 6000]
for x in cutoffs:
    disease_mappings, folds, train_dataset, test_dataset, test_loader = prepare_df(x)
    num_classes = len(disease_mappings)
    save_dir = f"logs_kfold_fastgan_topk_cos/SWIN_T/top{x}"
    device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
    backbone = models.swin_t(weights="Swin_T_Weights.IMAGENET1K_V1")
    model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False)
    train_model_on_folds(model, train_dataset, folds, save_dir, device, num_epochs=50)


In [ ]:
# swin_t test
cutoffs = [1000, 2000, 4000, 6000]
for x in cutoffs:
    disease_mappings, _, _, _, test_loader = prepare_df(x)
    test_metrics = []
    print(f"==========================cutoff {x}==========================")
    for fold in range(5):
        num_classes = len(disease_mappings)
        device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
        backbone = models.swin_t(weights="Swin_T_Weights.IMAGENET1K_V1")
        test_model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False)
        test_model.load_state_dict(torch.load(f'logs_kfold_fastgan_topk_cos/SWIN_T/top{x}/fold{fold+1}_best.pth'))
        test_model.to(device)
        test_model.eval()  # Set model to evaluation mode
        criterion = nn.CrossEntropyLoss()
        val_loss, val_acc, val_f1, topk_accuracies, _, _, _ = evaluate(test_model, test_loader, criterion, device)
        print(val_acc)
        print(" | ".join([f"Top-{k} Accuracy: {acc:.4f}%" for k, acc in topk_accuracies.items()]))

        test_metrics.append({
            "val_acc": val_acc,
            "val_topk": topk_accuracies
        })

    # Compute average and std of overall accuracy
    val_accs = [m["val_acc"] for m in test_metrics]
    avg_acc = np.mean(val_accs)
    std_acc = np.std(val_accs, ddof=1)

    # Compute average and std for each top-k accuracy
    avg_topk = {}
    std_topk = {}
    for k in test_metrics[0]["val_topk"].keys():
        scores = [m["val_topk"][k] for m in test_metrics]
        avg_topk[k] = np.mean(scores)
        std_topk[k] = np.std(scores, ddof=1)

    # Print results
    print(f"\nAverage across {len(test_metrics)} folds:")
    print(f" - Accuracy: {avg_acc:.2f}% ± {std_acc:.2f}% (1-sigma)")

    for k in sorted(avg_topk.keys(), key=int):
        print(f" - Top-{k} Accuracy: {avg_topk[k]:.2f}% ± {std_topk[k]:.2f}% (1-sigma)")

In [ ]:
# CLIP
cutoffs = [1000, 2000, 4000, 6000]
for x in cutoffs:
    disease_mappings, folds, train_dataset, test_dataset, test_loader = prepare_df(x)
    num_classes = len(disease_mappings)
    save_dir = f"logs_kfold_fastgan_topk_cos/CLIP/top{x}"
    device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
    clip_model, _= clip.load("ViT-B/32", device=device, jit=False)
    clip_model.visual = clip_model.visual.float()
    backbone = clip_model.visual
    model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False)
    train_model_on_folds(model, train_dataset, folds, save_dir, device, num_epochs=50)


In [ ]:
# clip test
cutoffs = [1000, 2000, 4000, 6000]
for x in cutoffs:
    disease_mappings, _, _, _, test_loader = prepare_df(x)
    test_metrics = []
    print(f"==========================cutoff {x}==========================")
    test_metrics = []
    for fold in range(5):
        num_classes = len(disease_mappings)
        device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
        clip_model, _= clip.load("ViT-B/32", device=device, jit=False)
        clip_model.visual = clip_model.visual.float()
        backbone = clip_model.visual
        test_model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False)
        test_model.load_state_dict(torch.load(f'logs_kfold_fastgan_topk_cos/CLIP/top{x}/fold{fold+1}_best.pth'))
        test_model.to(device)
        test_model.eval()  # Set model to evaluation mode
        criterion = nn.CrossEntropyLoss()
        val_loss, val_acc, val_f1, topk_accuracies, _, _, _ = evaluate(test_model, test_loader, criterion, device)
        print(val_acc)
        print(" | ".join([f"Top-{k} Accuracy: {acc:.4f}%" for k, acc in topk_accuracies.items()]))

        test_metrics.append({
            "val_acc": val_acc,
            "val_topk": topk_accuracies
        })

    # Compute average and std of overall accuracy
    val_accs = [m["val_acc"] for m in test_metrics]
    avg_acc = np.mean(val_accs)
    std_acc = np.std(val_accs, ddof=1)

    # Compute average and std for each top-k accuracy
    avg_topk = {}
    std_topk = {}
    for k in test_metrics[0]["val_topk"].keys():
        scores = [m["val_topk"][k] for m in test_metrics]
        avg_topk[k] = np.mean(scores)
        std_topk[k] = np.std(scores, ddof=1)

    # Print results
    print(f"\nAverage across {len(test_metrics)} folds:")
    print(f" - Accuracy: {avg_acc:.2f}% ± {std_acc:.2f}% (1-sigma)")

    for k in sorted(avg_topk.keys(), key=int):
        print(f" - Top-{k} Accuracy: {avg_topk[k]:.2f}% ± {std_topk[k]:.2f}% (1-sigma)")

In [ ]:
# VGGFACE
cutoffs = [1000, 2000, 4000, 6000]
for x in cutoffs:
    disease_mappings, folds, train_dataset, test_dataset, test_loader = prepare_df(x)
    num_classes = len(disease_mappings)
    save_dir = f"logs_kfold_fastgan_topk_cos/VGGFACE/top{x}"
    device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
    vgg = vgg_face('models_vggface/vgg_face.pth')
    backbone = VggFaceFeatureExtractor(vgg)
    model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False, conv_out_channels=512)
    train_model_on_folds(model, train_dataset, folds, save_dir, device, num_epochs=50)


In [ ]:
# vggface test
cutoffs = [1000, 2000, 4000, 6000]
for x in cutoffs:
    disease_mappings, _, _, _, test_loader = prepare_df(x)
    test_metrics = []
    print(f"==========================cutoff {x}==========================")
    for fold in range(5):
        num_classes = len(disease_mappings)
        device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
        vgg = vgg_face('models_vggface/vgg_face.pth')
        backbone = VggFaceFeatureExtractor(vgg)
        test_model = RareDiseaseModel(backbone=backbone, num_classes=num_classes, use_se=True, dropout_rate=0.5, use_layernorm=False, conv_out_channels=512)
        test_model.load_state_dict(torch.load(f'logs_kfold_fastgan_topk_cos/VGGFACE/top{x}/fold{fold+1}_best.pth'))
        test_model.to(device)
        test_model.eval()  # Set model to evaluation mode
        criterion = nn.CrossEntropyLoss()
        val_loss, val_acc, val_f1, topk_accuracies, _, _, _ = evaluate(test_model, test_loader, criterion, device)
        print(val_acc)
        print(" | ".join([f"Top-{k} Accuracy: {acc:.4f}%" for k, acc in topk_accuracies.items()]))

        test_metrics.append({
            "val_acc": val_acc,
            "val_topk": topk_accuracies
        })

    # Compute average and std of overall accuracy
    val_accs = [m["val_acc"] for m in test_metrics]
    avg_acc = np.mean(val_accs)
    std_acc = np.std(val_accs, ddof=1)

    # Compute average and std for each top-k accuracy
    avg_topk = {}
    std_topk = {}
    for k in test_metrics[0]["val_topk"].keys():
        scores = [m["val_topk"][k] for m in test_metrics]
        avg_topk[k] = np.mean(scores)
        std_topk[k] = np.std(scores, ddof=1)

    # Compute average and std of overall accuracy
    val_accs = [m["val_acc"] for m in test_metrics]
    avg_acc = np.mean(val_accs)
    std_acc = np.std(val_accs, ddof=1)

    # Compute average and std for each top-k accuracy
    avg_topk = {}
    std_topk = {}
    for k in test_metrics[0]["val_topk"].keys():
        scores = [m["val_topk"][k] for m in test_metrics]
        avg_topk[k] = np.mean(scores)
        std_topk[k] = np.std(scores, ddof=1)

    # Print results
    print(f"\nAverage across {len(test_metrics)} folds:")
    print(f" - Accuracy: {avg_acc:.2f}% ± {std_acc:.2f}% (1-sigma)")

    for k in sorted(avg_topk.keys(), key=int):
        print(f" - Top-{k} Accuracy: {avg_topk[k]:.2f}% ± {std_topk[k]:.2f}% (1-sigma)")